# Eksplorasi Dataset Petrobras 3W
**Capstone Halliburton Sperry Drilling**

Notebook ini mengeksplorasi dataset 3W (github.com/petrobras/3W).
Dataset path default: `../data/3W-main/dataset` atau `../data/3W/dataset` — sesuaikan.

In [ ]:
from pathlib import Path
import pandas as pd

# Path dataset — sesuaikan jika perlu
for p in [
    Path("../data/3W-main/dataset"),
    Path("../data/3W/dataset"),
]:
    if p.exists():
        DATASET_ROOT = p
        break
else:
    raise FileNotFoundError("Dataset 3W belum ada. Ekstrak dulu data/3W.zip")

print("Dataset root:", DATASET_ROOT)
print("exists:", DATASET_ROOT.exists())

## 1. Struktur kelas

In [ ]:
class_dirs = sorted([p for p in DATASET_ROOT.iterdir() if p.is_dir() and p.name.isdigit()])
print(f"Kelas: {len(class_dirs)}")
for d in class_dirs:
    n = len(list(d.glob("*.parquet")))
    print(f"  class {d.name}: {n} file")

## 2. Sampel file

In [ ]:
for cls in ["4", "0"]:
    files = sorted((DATASET_ROOT / cls).glob("*.parquet"))
    if not files:
        print(f"class {cls}: kosong")
        continue
    df = pd.read_parquet(files[0])
    print(f"--- {files[0].name} (class {cls}) ---")
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    print("class unique:", df["class"].unique())
    print()

## 3. Visualisasi sinyal P-TPT & T-TPT

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

for cls, ax in zip(["4", "0"], axes):
    files = sorted((DATASET_ROOT / cls).glob("*.parquet"))
    df = pd.read_parquet(files[0], columns=["P-TPT", "T-TPT", "class"])
    p = pd.to_numeric(df["P-TPT"], errors="coerce") / 100_000  # Pa -> bar
    t = pd.to_numeric(df["T-TPT"], errors="coerce")
    ax.plot(p.index, p, label="P-TPT (bar)", lw=0.7)
    ax2 = ax.twinx()
    ax2.plot(t.index, t, label="T-TPT", color="orange", lw=0.7)
    ax.set_title(f"class {cls} — {files[0].name}")
    ax.legend(loc="upper left")
    ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()

## 4. Hitung rate of temperature change (dT/dt)

In [ ]:
files4 = sorted((DATASET_ROOT / "4").glob("*.parquet"))
df = pd.read_parquet(files4[0], columns=["T-TPT", "class"])
t = pd.to_numeric(df["T-TPT"], errors="coerce")
dt = t.diff() / 1.0  # per detik (data 1 Hz)

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(dt.index, dt, lw=0.5)
ax.set_title("Rate of temperature change (dT/dt) — class 4")
ax.set_ylabel("degC/s")
plt.tight_layout()
plt.show()

print("dT/dt describe:")
print(dt.describe())